## General Preprocessing

In [47]:
import pandas as pd

DATASET = "BACNDA"

df1 = pd.read_csv(f"{DATASET}_0_49_summary.csv")
df2 = pd.read_csv(f"{DATASET}_50_99_summary.csv")

combined = pd.concat([df1, df2], ignore_index=True)

print(f"Combined {len(df1)} + {len(df2)} = {len(combined)} rows")
combined.head()

Combined 46 + 50 = 96 rows


,trial_idx,seed,best_epoch,n_epochs,d_model,d_ff_multiplier,d_ff,num_layers,num_heads,learning_rate,...,test_mean_abs_length_diff,test_mean_too_early,test_mean_too_late,test_MAE_rrt_stand,test_MAE_rrt_minutes,val_MAE_ttne_stand,val_MAE_ttne_minutes,val_DL_similarity,val_MAE_rrt_stand,val_MAE_rrt_minutes
0,0,42,52,73,48,5,240,2,12,0.000281,...,1.175047,3.024654,3.268782,0.342815,5.166638,0.351738,1.733559,0.744258,0.303330,4.570669
1,1,43,97,100,16,7,112,2,4,0.000016,...,2.048705,2.683926,9.629459,0.369798,5.567560,0.392762,1.932097,0.694718,0.333139,5.013729
2,2,44,44,46,64,3,192,4,16,0.000084,...,1.181731,2.997199,3.798484,0.340061,5.126652,0.345644,1.702702,0.746845,0.300851,4.534562
3,3,45,8,24,64,5,320,5,16,0.003975,...,2.090746,3.121765,10.225901,0.351355,5.291219,0.367319,1.808728,0.704605,0.311026,4.682224
4,4,46,81,100,48,7,336,5,12,0.002181,...,1.237961,3.070006,3.515426,0.348022,5.252868,0.358185,1.766465,0.730832,0.306965,4.633133


In [48]:
cols_to_drop = ["trial_idx", "d_ff", "num_heads","seed", "best_epoch", "n_epochs",
                "val_MAE_ttne_stand", "val_MAE_ttne_minutes", "val_DL_similarity",
                "val_MAE_rrt_stand", "val_MAE_rrt_minutes","test_MAE_ttne_stand", "test_perc_too_early", "test_perc_too_late",
                 "test_perc_correct", "test_mean_abs_length_diff", "test_mean_too_early",
                 "test_mean_too_late", "test_MAE_rrt_stand"]

combined = combined.drop(columns=cols_to_drop)
combined.to_csv(f"{DATASET}_combined.csv", index=False)

print(combined.columns.tolist())
combined.head()

['d_model', 'd_ff_multiplier', 'num_layers', 'learning_rate', 'dropout', 'weight_decay', 'test_MAE_ttne_minutes', 'test_DL_similarity', 'test_MAE_rrt_minutes']


,d_model,d_ff_multiplier,num_layers,learning_rate,dropout,weight_decay,test_MAE_ttne_minutes,test_DL_similarity,test_MAE_rrt_minutes
0,48,5,2,0.000281,0.342341,0.004081,1.844114,0.740116,5.166638
1,16,7,2,0.000016,0.438404,0.000419,2.009759,0.693721,5.567560
2,64,3,4,0.000084,0.066118,0.000030,1.810516,0.742513,5.126652
3,64,5,5,0.003975,0.042205,0.003574,1.928357,0.700739,5.291219
4,48,7,5,0.002181,0.642920,0.002948,1.883787,0.726292,5.252868


In [49]:
hyperparam_cols = ["d_model", "d_ff_multiplier", "num_layers", "learning_rate", "dropout", "weight_decay"]

df_ttne = combined[hyperparam_cols + ["test_MAE_ttne_minutes"]]
df_dl   = combined[hyperparam_cols + ["test_DL_similarity"]]
df_rrt  = combined[hyperparam_cols + ["test_MAE_rrt_minutes"]]

df_ttne.to_csv(f"{DATASET}_ttne.csv", index=False)
df_dl.to_csv(f"{DATASET}_DL.csv", index=False)
df_rrt.to_csv(f"{DATASET}_rrt.csv", index=False)

print("TTNE:", df_ttne.shape)
print("DL:  ", df_dl.shape)
print("RRT: ", df_rrt.shape)


TTNE: (96, 7)
DL:   (96, 7)
RRT:  (96, 7)


## Performance Capping

In [50]:
import numpy as np

TARGET = "DL"

FILE = f"{DATASET}_{TARGET}.csv"

# Percentile at which to cap (e.g. 25 means 25th percentile)
PERCENTILE = 60

if TARGET == "DL":
    CAP_DIRECTION = "floor"   
else:
    CAP_DIRECTION = "ceiling"

# Output file name
OUTPUT_FILE = f"{DATASET}_{TARGET}_capped.csv"

In [51]:
df = pd.read_csv(FILE)
print(f"Loaded {len(df)} rows from '{FILE}'")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

Loaded 96 rows from 'BACNDA_DL.csv'

Columns: ['d_model', 'd_ff_multiplier', 'num_layers', 'learning_rate', 'dropout', 'weight_decay', 'test_DL_similarity']


,d_model,d_ff_multiplier,num_layers,learning_rate,dropout,weight_decay,test_DL_similarity
0,48,5,2,0.000281,0.342341,0.004081,0.740116
1,16,7,2,0.000016,0.438404,0.000419,0.693721
2,64,3,4,0.000084,0.066118,0.000030,0.742513
3,64,5,5,0.003975,0.042205,0.003574,0.700739
4,48,7,5,0.002181,0.642920,0.002948,0.726292


In [52]:
HYPERPARAM_COLS = ["d_model", "d_ff_multiplier", "num_layers",
                   "learning_rate", "dropout", "weight_decay"]

performance_cols = [c for c in df.columns if c not in HYPERPARAM_COLS]
print(f"Performance column(s) detected: {performance_cols}")

df.describe()

Performance column(s) detected: ['test_DL_similarity']


,d_model,d_ff_multiplier,num_layers,learning_rate,dropout,weight_decay,test_DL_similarity
count,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000
mean,44.083333,5.010417,5.000000,0.001496,0.353843,0.001486,0.685437
std,30.090274,1.785879,1.771113,0.002325,0.200909,0.002303,0.144811
min,8.000000,2.000000,2.000000,0.000010,0.006696,0.000011,0.042119
25%,16.000000,3.750000,3.750000,0.000053,0.183007,0.000061,0.695086
50%,40.000000,5.000000,5.000000,0.000318,0.358086,0.000332,0.721148
75%,64.000000,6.250000,6.250000,0.001912,0.520027,0.001875,0.735783
max,96.000000,8.000000,8.000000,0.009808,0.695759,0.009418,0.748252


In [53]:
df_capped = df.copy()

for col in performance_cols:
    threshold = np.percentile(df[col], PERCENTILE)
    
    if CAP_DIRECTION == "floor":
        df_capped[col] = df[col].clip(lower=threshold)
        n_capped = (df[col] < threshold).sum()
    elif CAP_DIRECTION == "ceiling":
        df_capped[col] = df[col].clip(upper=threshold)
        n_capped = (df[col] > threshold).sum()
    else:
        raise ValueError("CAP_DIRECTION must be 'floor' or 'ceiling'")
    
    print(f"\nColumn : {col}")
    print(f"  Percentile  : {PERCENTILE}th  →  threshold = {threshold:.6f}")
    print(f"  Direction   : {CAP_DIRECTION}")
    print(f"  Rows capped : {n_capped} / {len(df)} ({100*n_capped/len(df):.1f}%)")
    print(f"  Before — min: {df[col].min():.4f}, mean: {df[col].mean():.4f}, max: {df[col].max():.4f}")
    print(f"  After  — min: {df_capped[col].min():.4f}, mean: {df_capped[col].mean():.4f}, max: {df_capped[col].max():.4f}")

df_capped.head()


Column : test_DL_similarity
  Percentile  : 60th  →  threshold = 0.730334
  Direction   : floor
  Rows capped : 57 / 96 (59.4%)
  Before — min: 0.0421, mean: 0.6854, max: 0.7483
  After  — min: 0.7303, mean: 0.7333, max: 0.7483


,d_model,d_ff_multiplier,num_layers,learning_rate,dropout,weight_decay,test_DL_similarity
0,48,5,2,0.000281,0.342341,0.004081,0.740116
1,16,7,2,0.000016,0.438404,0.000419,0.730334
2,64,3,4,0.000084,0.066118,0.000030,0.742513
3,64,5,5,0.003975,0.042205,0.003574,0.730334
4,48,7,5,0.002181,0.642920,0.002948,0.730334


In [54]:
# Reorder columns: hyperparameters alphabetically, performance columns last
sorted_hyperparam_cols = sorted(HYPERPARAM_COLS)
df_capped = df_capped[sorted_hyperparam_cols + performance_cols]
print(f"Column order: {df_capped.columns.tolist()}")

Column order: ['d_ff_multiplier', 'd_model', 'dropout', 'learning_rate', 'num_layers', 'weight_decay', 'test_DL_similarity']


In [55]:
df_capped.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(df_capped)} rows to '{OUTPUT_FILE}'")

Saved 96 rows to 'BACNDA_DL_capped.csv'
